In [13]:
from autoresearch_problems import (
    registry,
    execute_and_evaluate,
    execute_and_evaluate_batch,
    run_evaluation
)

from autoresearch_problems.program_runners import SubprocessRunner
import numpy as np

In [ ]:
spec = registry.load('analysis/first_autocorrelation_inequality')
result = execute_and_evaluate(spec, spec.initial_program, function_name='solve')
print(f'Single candidate: score={result.score}, valid={result.valid}')

In [15]:
spec = registry.load('analysis/first_autocorrelation_inequality')
initial_prg = spec.initial_program

print("Initial program:")
print(initial_prg)
runner = SubprocessRunner(timeout=spec.timeout_seconds)
try:
    candidate_output = runner.execute(spec.initial_program, function_name='solve')
except Exception as e:
    print(f"Error executing initial program: {e}")


result = run_evaluation(spec, candidate_output)

Initial program:
"""Naive uniform seed solution for the First Autocorrelation Inequality problem."""


def solve() -> list[float]:
    """Return a uniform step function on 600 equally-spaced intervals.

    This is the simplest possible baseline: f(x) = 1 everywhere on [-1/4, 1/4].
    The resulting C₁ ratio is the starting point for the LLM to improve upon.
    """
    num_intervals = 600
    return [1.0] * num_intervals



In [6]:
import numpy as np
from typing import Any, Dict, List, Tuple

BENCHMARK = 1.5052939684401607

def validate_sequence(run_output: Any, atol=1e-6) -> Tuple[bool, str]:
    """
    Validates the sequence returned by the evolved program.
    Mimics the checks in the original evaluate_sequence function.
    """
    sequence = run_output

    if not isinstance(sequence, list):
        return False, f"Sequence type expected to be list, received {type(sequence)}"

    if not sequence:
        return False, "Sequence cannot be None or empty."

    # Type and value checks
    for x in sequence:
        if isinstance(x, bool) or not isinstance(x, (int, float, np.number)):
             return False, "Sequence entries must be integers or floats."
        if np.isnan(x) or np.isinf(x):
             return False, "Sequence cannot contain nans or infs."
        #if x == 0.0:
        #     return False, "Sequence cannot contain zero entries."

    # Simulate the transformation to check the sum constraint
    try:
        seq_array = np.array([float(x) for x in sequence])
        seq_array = np.maximum(0, seq_array)
        seq_array = np.minimum(10000000.0, seq_array)
        
        sum_a = np.sum(seq_array)
        if sum_a < 0.00001:
            return False, f"Sum of sequence entries too close to zero: {sum_a}."
            
    except Exception as e:
        return False, f"Error during validation processing: {str(e)}"

    return True, "Sequence is valid."

def get_experiment_kwargs(run_index: int) -> Dict[str, Any]:
    return {}


def compute_c1(sequence: List[float]) -> float:
    sequence = [float(x) for x in sequence]
    sequence = [max(0, x) for x in sequence]
    sequence = [min(10000000.0, x) for x in sequence]

    n = len(sequence)
    b_sequence = np.convolve(sequence, sequence)
    max_b = max(b_sequence)
    sum_a = np.sum(sequence)
    
    # Calculate c1 as per original script
    c1 = float(2 * n * max_b / (sum_a**2))

    return c1

In [ ]:
k

In [27]:
import numpy as np
import time
from scipy import optimize
linprog = optimize.linprog


def solve_convolution_lp(f_sequence, rhs):
    """Solves the convolution LP for a given sequence and RHS."""
    n = len(f_sequence)
    c = -np.ones(n)
    a_ub = []
    b_ub = []
    for k in range(2 * n - 1):
        row = np.zeros(n)
        for i in range(n):
            j = k - i
            if 0 <= j < n:
                row[j] = f_sequence[i]
        a_ub.append(row)
        b_ub.append(rhs)

    # Non-negativity constraints: b_i >= 0
    a_ub_nonneg = -np.eye(n)  # Negative identity matrix for b_i >= 0
    b_ub_nonneg = np.zeros(n)  # Zero vector

    a_ub = np.vstack([a_ub, a_ub_nonneg])
    b_ub = np.hstack([b_ub, b_ub_nonneg])

    result = linprog(c, A_ub=a_ub, b_ub=b_ub)

    if result.success:
        g_sequence = result.x
        return g_sequence
    else:
        print('LP optimization failed.')
        return None


def get_good_direction_to_move_into(sequence: list[float]) -> list[float] | None:
    """Returns the direction to move into the sequence."""
    n = len(sequence)
    sum_sequence = np.sum(sequence)
    normalized_sequence = [x * np.sqrt(2 * n) / sum_sequence for x in sequence]
    rhs = np.max(np.convolve(normalized_sequence, normalized_sequence))
    g_fun = solve_convolution_lp(normalized_sequence, rhs)
    if g_fun is None:
        return None
    sum_sequence = np.sum(g_fun)
    normalized_g_fun = [x * np.sqrt(2 * n) / sum_sequence for x in g_fun]
    t = 0.01
    new_sequence = [(1 - t) * x + t * y for x, y in zip(sequence, normalized_g_fun)]
    return new_sequence


def run() -> list[float]:
    """Function to search for the best coefficient sequence."""
    best_sequence_prev = get_best_sequence_prev()
    if np.random.rand() < 0.5 and best_sequence_prev:
        best_sequence = best_sequence_prev
    else:
        best_sequence = [np.random.random()] * np.random.randint(100,1000)
    curr_sequence = best_sequence.copy()
    best_score = np.inf
    start_time = time.time()
    while time.time() - start_time < 1000:
        h_function = get_good_direction_to_move_into(curr_sequence)
        if h_function is None:
            curr_sequence[1] = (curr_sequence[1] + np.random.rand()) % 1
        else:
            curr_sequence = h_function

        curr_score = evaluate_sequence(curr_sequence)
        if curr_score < best_score:
            best_score = curr_score
            best_sequence = curr_sequence

    return best_sequence

# EVOLVE-BLOCK-END


def evaluate_sequence(sequence: list[float]) -> float:
  
  if not isinstance(sequence, list):
    return np.inf
  # Reject empty lists
  if not sequence:
    return np.inf

  # Check each element in the list for validity
  for x in sequence:
    # Reject boolean types (as they are a subclass of int) and
    # any other non-integer/non-float types (like strings or complex numbers).
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return np.inf

    # Reject Not-a-Number (NaN) and infinity values.
    if np.isnan(x) or np.isinf(x):
        return np.inf

  # Convert all elements to float for consistency
  sequence = [float(x) for x in sequence]

  # Protect against negative numbers
  sequence = [max(0, x) for x in sequence]

  # Protect against numbers that are too large
  sequence = [min(1000.0, x) for x in sequence]

  n = len(sequence)
  b_sequence = np.convolve(sequence, sequence)
  max_b = max(b_sequence)
  sum_a = np.sum(sequence)

  # Protect against the case where the sum is too close to zero
  if sum_a < 0.01:
    return np.inf

  return float(2 * n * max_b / (sum_a**2))


# Reads file best_sequence_prev.txt and returns the sequence as a list of floats.
def get_best_sequence_prev() -> list[float]:
    with open("best_sequence_prev_alpha_evolve.txt", "r") as f:
        line = f.readline().strip()
        if not line:
            return []
        return [float(x) for x in line.split(",")]

In [28]:
candidate_output = run()

FileNotFoundError: [Errno 2] No such file or directory: 'best_sequence_prev_alpha_evolve.txt'

In [22]:
compute_c1(candidate_output)

1.999998530371408

In [24]:
len(candidate_output)

783

In [23]:
run_evaluation(spec, candidate_output)

EvalResult(score=1.999998530371408, valid=True, execution_time=0.002978033386170864, error='', metrics={'c1': 1.999998530371408, 'sequence_length': 783, 'integral_f': 0.294791748259994, 'max_autoconv': 0.17380422197044648})